# Занятие 6. Списки, кортежи, цена операции

**План занятия**

1. Операции списка
2. `sort` против `sorted`, параметр `key`
3. Список это ссылка
4. Кортеж
5. Цена операции и O-нотация
6. Как увидеть квадрат в своём коде
7. Домашние задачи

**Теория:** `theory/06_Списки_и_цена_операции.md`
Т. Гэддис, гл. 7 (с. 365 / PDF 390)
А. Бхаргава, гл. 1 (с. 18 / PDF 19) и гл. 2 (с. 40 / PDF 41)

In [ ]:
# Эта ячейка находит папку с данными. Запустите её ПЕРВОЙ.
import os

CANDIDATES = ["../data", "data", "./data", "/content/data",
              "/content/drive/MyDrive/mglu/data"]
DATA = next((p for p in CANDIDATES if os.path.isdir(p)), None)
print("Data folder:", os.path.abspath(DATA) if DATA else "NOT FOUND")

---

## 1. Операции списка

In [ ]:
items = ["a", "b", "c"]

print(items[0], items[-1], items[0:2], len(items))

items.append("d")
print("append:", items)

items.insert(0, "z")
print("insert(0):", items)

items.remove("b")
print("remove:", items)

last = items.pop()
print("pop:", last, items)

### `sort` против `sorted`

**Проверьте себя.** Что напечатают обе строки?

In [ ]:
items = ["c", "a", "b"]
result = items.sort()

print("вернул:", result)
print("список:", items)

In [ ]:
items = ["c", "a", "b"]
items = items.sort()          # так теряют данные
print(items)

In [ ]:
items = ["c", "a", "b"]
ordered = sorted(items)       # копия, исходный цел
print(ordered, items)

### Параметр `key`

In [ ]:
pairs = []
with open(f"{DATA}/frequencies.csv", encoding="utf-8") as f:
    f.readline()
    for line in f:
        word, count = line.rstrip("\n").split(",")
        pairs.append((word, int(count)))

print("пар:", len(pairs))
print(pairs[:5])

In [ ]:
top = sorted(pairs, key=lambda pair: pair[1], reverse=True)[:10]
for word, count in top:
    print(f"{word:>15} {count:>5}")

In [ ]:
# два критерия сразу: по длине слова, при равной длине по убыванию частоты
by_length = sorted(pairs, key=lambda pair: (len(pair[0]), -pair[1]))
print("короткие:", by_length[:5])
print("длинные: ", by_length[-5:])

Кортеж в `key` означает сортировку по нескольким критериям. Минус перед числом
переворачивает направление для одного критерия.

---

## 2. Список это ссылка

In [ ]:
a = ["x", "y"]
b = a                # второе имя для того же объекта
b.append("z")

print("a:", a)
print("b:", b)
print("один объект:", a is b)

In [ ]:
a = ["x", "y"]
b = a.copy()         # или a[:], или list(a)
b.append("z")

print("a:", a)
print("b:", b)
print("один объект:", a is b)

Одна из немногих ошибок курса, которая портит данные без сообщения.

---

## 3. Кортеж

In [ ]:
record = ("Pushkin", 1799, "poet")
name, year, role = record
print(name, year, role)

try:
    record[1] = 1800
except TypeError as error:
    print("TypeError:", error)

In [ ]:
def stats(text):
    words = text.split()
    return len(words), len(set(words))       # вернули кортеж

with open(f"{DATA}/text_clean.txt", encoding="utf-8") as f:
    total, unique = stats(f.read())

print(f"всего {total}, уникальных {unique}, отношение {unique / total:.3f}")

---

## 4. Как это лежит в памяти

`list` в Python это массив: элементы подряд, доступ по индексу мгновенный,
вставка в середину сдвигает всё правее.

In [ ]:
import time

N = 50_000

start = time.perf_counter()
a = []
for i in range(N):
    a.append(i)                 # в конец, O(1)
append_time = time.perf_counter() - start

start = time.perf_counter()
b = []
for i in range(N):
    b.insert(0, i)              # в начало, O(n)
insert_time = time.perf_counter() - start

print(f"append:    {append_time:.4f} s")
print(f"insert(0): {insert_time:.4f} s")
print(f"ratio: {insert_time / append_time:.0f}x")

---

## 5. Цена операции

Скорость измеряют ростом работы при росте данных.

In [ ]:
with open(f"{DATA}/words_ru.txt", encoding="utf-8") as f:
    word_list = [w.strip() for w in f if w.strip()]

word_set = set(word_list)
print("слов:", len(word_list))

In [ ]:
queries = [f"missing{i}" for i in range(3000)]     # худший случай, перебор до конца

start = time.perf_counter()
for q in queries:
    q in word_list                  # O(n)
list_time = time.perf_counter() - start

start = time.perf_counter()
for q in queries:
    q in word_set                   # O(1)
set_time = time.perf_counter() - start

print(f"list: {list_time:.4f} s")
print(f"set:  {set_time:.6f} s")
print(f"ratio: {list_time / set_time:.0f}x")

Одного замера мало. Посмотрим, как разница растёт.

In [ ]:
print(f"{'size':>8} {'list, s':>10} {'set, s':>12} {'ratio':>8}")
for size in [1000, 2000, 4000, 8000]:
    lst = [f"w{i}" for i in range(size)]
    st = set(lst)
    qs = [f"q{i}" for i in range(2000)]

    start = time.perf_counter()
    for q in qs:
        q in lst
    t_list = time.perf_counter() - start

    start = time.perf_counter()
    for q in qs:
        q in st
    t_set = time.perf_counter() - start

    print(f"{size:>8} {t_list:>10.4f} {t_set:>12.6f} {t_list / t_set:>8.0f}")

Данные выросли вдвое, время списка выросло вдвое. Это O(n).
Данные выросли в 8 раз, время множества не изменилось. Это O(1).

| n | O(n) | O(n log n) | O(n²) |
|---|---|---|---|
| 1 000 | 0,001 с | 0,01 с | 1 с |
| 100 000 | 0,1 с | 1,7 с | 3 часа |
| 1 000 000 | 1 с | 20 с | 11 дней |

Таблица составлена для операции длительностью в микросекунду.
O(n²) означает «невозможно», а не «медленно».

---

## 6. Как увидеть квадрат

Три признака.

In [ ]:
# 1. `in` по списку внутри цикла
def unique_slow(items):
    result = []
    for item in items:
        if item not in result:      # здесь спрятан второй цикл
            result.append(item)
    return result

def unique_fast(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:        # O(1)
            seen.add(item)
            result.append(item)
    return result

data = [f"w{i % 3000}" for i in range(12000)]

start = time.perf_counter()
slow = unique_slow(data)
t1 = time.perf_counter() - start

start = time.perf_counter()
fast = unique_fast(data)
t2 = time.perf_counter() - start

print(f"slow: {t1:.4f} s")
print(f"fast: {t2:.4f} s")
print("равны:", slow == fast)

In [ ]:
# 2. коллекция пересобирается на каждом шаге
def filter_slow(items, stop):
    result = []
    for item in items:
        if item not in set(stop):       # множество строится ЗАНОВО каждый раз
            result.append(item)
    return result

def filter_fast(items, stop):
    stop_set = set(stop)                # один раз ДО цикла
    return [x for x in items if x not in stop_set]

stop = [f"s{i}" for i in range(2000)]
data = [f"w{i}" for i in range(5000)]

start = time.perf_counter()
filter_slow(data, stop)
t1 = time.perf_counter() - start

start = time.perf_counter()
filter_fast(data, stop)
t2 = time.perf_counter() - start

print(f"пересборка в цикле: {t1:.4f} s")
print(f"один раз до цикла:  {t2:.5f} s")

Второй случай коварнее прочих. Он выглядит как забота о корректности,
а множество строится заново на каждой итерации.

| Мне нужно | Структура |
|---|---|
| знать порядок, обращаться по номеру | `list` |
| проверять наличие много раз | `set` |
| связать одно с другим | `dict`, занятие 7 |
| хранить запись из разных частей | `tuple` |
| часто добавлять в начало | `deque`, занятие 24 |

---

# Домашние задачи

Рассчитаны примерно на 30 минут.

### Задача 1. Трассировка (без запуска)

Что напечатает код?

In [ ]:
# Мой ответ: ...

# a = [1, 2, 3]
# b = a
# c = a.copy()
# a.append(4)
# print(b, c)
# print(a is b, a is c)
# print(a == c)

### Задача 2. Минимум LeetCode

**LeetCode 977 Squares of a Sorted Array** плюс письменный разбор.

Решите как получится, скорее всего сортировкой. Запишите сложность своего решения.
На занятии 21 вернёмся к этой задаче и решим за один проход.

In [ ]:
def sorted_squares(nums):
    # ваш код здесь
    pass

# print(sorted_squares([-4, -1, 0, 3, 10]))

### Задача 3. Две реализации и замер

**LeetCode 217 Contains Duplicate**, главная задача занятия.
Решите дважды и замерьте на входе без дубликатов, это худший случай.

In [ ]:
def has_duplicate_list(nums):
    seen = []
    for n in nums:
        if n in seen:          # O(n)
            return True
        seen.append(n)
    return False

def has_duplicate_set(nums):
    # ваш код здесь
    pass

data = list(range(8000))       # дублей нет, перебор до конца

start = time.perf_counter()
has_duplicate_list(data)
print(f"через список: {time.perf_counter() - start:.4f} s")

# раскомментируйте, когда напишете вторую функцию
# start = time.perf_counter()
# has_duplicate_set(data)
# print(f"через множество: {time.perf_counter() - start:.6f} s")

### Задача 4. Сортировка по двум критериям

Отсортируйте `pairs` так: сначала по первой букве слова по алфавиту,
внутри буквы по убыванию частоты. Выведите первые 15.

In [ ]:
# ваш код здесь

### Задача 5. Найдите квадрат

Функция ищет слова, встречающиеся в обоих текстах. Здесь два разных источника
квадратичности. Найдите оба и перепишите за O(n).

In [ ]:
def common_words(text1, text2):
    result = []
    for w in text1.split():
        if w in text2.split():
            result.append(w)
    return result

def common_words_fast(text1, text2):
    # ваш код здесь
    pass

### Задача 6. Подумать (кода не нужно)

Программа обрабатывает 1000 документов за 2 секунды. Заказчик говорит,
что документов будет 100 000. Сколько это займёт при сложности O(n), O(n log n), O(n²)?
Какой из трёх ответов означает «надо переписывать»?

### Задача 7. Трек «алгоритмы» (по желанию)

283 Move Zeroes, 448 Find All Numbers Disappeared in an Array.

---

# Итоги

- `sort` меняет на месте и возвращает `None`, `sorted` возвращает копию.
- `b = a` даёт второе имя, копия делается через `copy()`.
- `key` с кортежем сортирует по нескольким критериям.
- `list` это массив: индекс дёшев, `insert(0)` дорог.
- `in` по списку линеен, по множеству мгновенен.
- O(n²) означает невозможность, а не медленность.
- Три признака квадрата: цикл в цикле, `in` по списку внутри цикла,
  пересборка коллекции на каждом шаге.